In [4]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from collections import defaultdict

In [2]:
SURVEY_CSV = (
    Path.cwd().parents[1]
    / "data"
    / "stack-overflow-developer-survey-2025"
    / "survey_results_public_2025.csv"
)

spark = SparkSession.builder.appName("so_survey_local").getOrCreate()

# Spark needs a file: URI or absolute path string
csv_path = SURVEY_CSV.as_posix()
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(csv_path)
)
df.select("ResponseId", "Country", "AISelect").show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 17:46:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----------+-----------+--------------------+
|ResponseId|    Country|            AISelect|
+----------+-----------+--------------------+
|         1|    Ukraine|Yes, I use AI too...|
|         2|Netherlands|Yes, I use AI too...|
|         3|    Ukraine|Yes, I use AI too...|
|         4|    Ukraine|Yes, I use AI too...|
|         5|    Ukraine|Yes, I use AI too...|
+----------+-----------+--------------------+
only showing top 5 rows


In [3]:
df_bronze = df
print(f"Rows: {df_bronze.count():,}")
print(f"Columns: {len(df_bronze.columns)}")
df_bronze.printSchema()

Rows: 49,191
Columns: 172
root
 |-- ResponseId: integer (nullable = true)
 |-- MainBranch: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- EdLevel: string (nullable = true)
 |-- Employment: string (nullable = true)
 |-- EmploymentAddl: string (nullable = true)
 |-- WorkExp: string (nullable = true)
 |-- LearnCodeChoose: string (nullable = true)
 |-- LearnCode: string (nullable = true)
 |-- LearnCodeAI: string (nullable = true)
 |-- AILearnHow: string (nullable = true)
 |-- YearsCode: string (nullable = true)
 |-- DevType: string (nullable = true)
 |-- OrgSize: string (nullable = true)
 |-- ICorPM: string (nullable = true)
 |-- RemoteWork: string (nullable = true)
 |-- PurchaseInfluence: string (nullable = true)
 |-- TechEndorseIntro: string (nullable = true)
 |-- TechEndorse_1: string (nullable = true)
 |-- TechEndorse_2: string (nullable = true)
 |-- TechEndorse_3: string (nullable = true)
 |-- TechEndorse_4: string (nullable = true)
 |-- TechEndorse_5: string (nullab

# Column Classification

Survey data has 172 columns but only **5 shapes**. Once we know the shape, we know how to transform it. This cell classifies every column automatically.

In [5]:
all_cols = df_bronze.columns

# ── 1. RANKING columns: groups of numbered columns sharing a prefix ──────────
# Pattern: TechEndorse_1, TechEndorse_2, ... or JobSatPoints_1, ...
# Values are integers representing rank position.
ranking_prefixes = [
    "TechEndorse_", "TechOppose_", "JobSatPoints_", "SO_Actions_",
]
ranking_cols = [c for c in all_cols if any(c.startswith(p) for p in ranking_prefixes)]

# ── 2. INVERTED MATRIX columns: response level is in the column name ─────────
# Pattern: AIToolCurrently partially AI, AIAgentImpactSomewhat agree, ...
# Values are semicolon-delimited lists of items.
inverted_matrix_prefixes = [
    "AITool",           # AIToolCurrently partially AI, AIToolPlan to mostly use AI, ...
    "AIAgentImpact",    # AIAgentImpactSomewhat agree, AIAgentImpactNeutral, ...
    "AIAgentChallenges", # AIAgentChallengesNeutral, AIAgentChallengesSomewhat agree, ...
]
inverted_matrix_cols = [
    c for c in all_cols
    if any(c.startswith(p) for p in inverted_matrix_prefixes)
    and c not in ranking_cols
]

# ── 3. HAVE / WANT / ADMIRED triples: multi-select semicolon-delimited ───────
# Pattern: LanguageHaveWorkedWith, LanguageWantToWorkWith, LanguageAdmired
triple_suffixes = ["HaveWorkedWith", "WantToWorkWith", "Admired"]
triple_cols = [c for c in all_cols if any(c.endswith(s) for s in triple_suffixes)]

# ── 4. OTHER MULTI-SELECT columns: semicolon-delimited but not triples ───────
# Identified by checking for semicolons in sample values.
remaining = set(all_cols) - set(ranking_cols) - set(inverted_matrix_cols) - set(triple_cols)

# Sample 200 rows to detect semicolons (cheap heuristic)
sample_rows = df_bronze.limit(200).toPandas()
other_multiselect_cols = []
for c in sorted(remaining):
    vals = sample_rows[c].dropna().astype(str)
    if vals.str.contains(";").any():
        other_multiselect_cols.append(c)

# ── 5. FREE TEXT columns: open-ended, write-ins, "Other" companions ──────────
# Three sub-types:
#   a) "Other (please specify)" companions to rankings/multi-selects (end in _TEXT)
#   b) Write-in entries for tech lists (end in Entry / Entr, Write)
#   c) Standalone open-ended questions (AIExplain, AIOpen)
free_text_suffixes = ("_TEXT", "Entry", "Entr", "Write")  # Entr: truncated export e.g. CommPlatformHaveEntr
standalone_free_text = {"AIExplain", "AIOpen"}
free_text_cols = [
    c for c in remaining
    if c.endswith(free_text_suffixes) or c in standalone_free_text
]

# ── 6. SCALAR columns: everything left ───────────────────────────────────────
classified = (
    set(ranking_cols)
    | set(inverted_matrix_cols)
    | set(triple_cols)
    | set(other_multiselect_cols)
    | set(free_text_cols)
)
scalar_cols = [c for c in all_cols if c not in classified]

# ── Print summary ─────────────────────────────────────────────────────────────
groups = {
    "SCALAR (single value per cell)": scalar_cols,
    "RANKING (integer rank, unpivot needed)": ranking_cols,
    "INVERTED MATRIX (column=response, value=semicolon items)": inverted_matrix_cols,
    "HAVE/WANT/ADMIRED triples (semicolon multi-select)": triple_cols,
    "OTHER MULTI-SELECT (semicolon-delimited)": other_multiselect_cols,
    "FREE TEXT (open-ended, write-ins, 'Other' companions)": free_text_cols,
}

for label, cols in groups.items():
    print(f"\n{'='*70}")
    print(f"{label}  [{len(cols)} columns]")
    print(f"{'='*70}")
    for c in cols:
        print(f"  {c}")

26/04/14 17:48:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



SCALAR (single value per cell)  [43 columns]
  ResponseId
  MainBranch
  Age
  EdLevel
  Employment
  WorkExp
  LearnCodeChoose
  LearnCodeAI
  YearsCode
  DevType
  OrgSize
  ICorPM
  RemoteWork
  PurchaseInfluence
  TechEndorseIntro
  Industry
  AIThreat
  NewRole
  ToolCountWork
  ToolCountPersonal
  Country
  Currency
  CompTotal
  LanguageChoice
  DatabaseChoice
  PlatformChoice
  WebframeChoice
  DevEnvsChoice
  AIModelsChoice
  SOAccount
  SOVisitFreq
  SODuration
  SOPartFreq
  SOComm
  SOFriction
  AISelect
  AISent
  AIAcc
  AIComplex
  AIAgents
  AIAgentChange
  ConvertedCompYearly
  JobSat

RANKING (integer rank, unpivot needed)  [49 columns]
  TechEndorse_1
  TechEndorse_2
  TechEndorse_3
  TechEndorse_4
  TechEndorse_5
  TechEndorse_6
  TechEndorse_7
  TechEndorse_8
  TechEndorse_9
  TechEndorse_13
  TechEndorse_13_TEXT
  TechOppose_1
  TechOppose_2
  TechOppose_3
  TechOppose_5
  TechOppose_7
  TechOppose_9
  TechOppose_11
  TechOppose_13
  TechOppose_16
  TechOppose_15

# Peek at each shape

Each shape needs a different transformation strategy. Let's see what they actually look like.

In [ ]:
# ── Shape 1: SCALAR ──────────────────────────────────────────────────────────
# These are the easy ones: one value per cell.
# Examples: Age, Country, JobSat, WorkExp, CompTotal
print("SCALAR examples (one value per cell — the easy ones):")
df_bronze.select("ResponseId", "Age", "Country", "WorkExp", "JobSat", "AISelect").show(5, truncate=40)

In [ ]:
# ── Shape 2: HAVE/WANT/ADMIRED (semicolon multi-select) ──────────────────────
# Each cell holds multiple values separated by semicolons.
# e.g. "Python;JavaScript;SQL" — needs exploding into separate rows.
print("HAVE/WANT/ADMIRED examples (semicolon-separated — need exploding):")
df_bronze.select(
    "ResponseId",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "LanguageAdmired",
).show(5, truncate=50)

In [ ]:
# ── Shape 3: INVERTED MATRIX (the weirdest shape) ────────────────────────────
# The COLUMN NAME contains the response level (e.g. "Currently partially AI").
# The CELL VALUE is a semicolon list of tasks that got that response.
# This is backwards from what you'd expect — needs unpivoting + exploding.
print("INVERTED MATRIX examples (column name = response, value = semicolon list of tasks):")
ai_tool_cols = [c for c in all_cols if c.startswith("AITool")]
df_bronze.select("ResponseId", *ai_tool_cols).show(3, truncate=60)

In [ ]:
# ── Shape 4: RANKINGS (integer rank per item) ────────────────────────────────
# Each column is one item in a ranked list. The value is the rank (1 = most important).
# e.g. TechEndorse_1 = rank of "AI integration" for that respondent.
# Needs unpivoting: wide columns → (respondent, item, rank) rows.
print("RANKING examples (each column = one item, value = rank position):")
tech_endorse_cols = [c for c in all_cols if c.startswith("TechEndorse_") and "TEXT" not in c]
df_bronze.select("ResponseId", *tech_endorse_cols).show(3, truncate=20)

In [ ]:
# ── Shape 5: OTHER MULTI-SELECT ──────────────────────────────────────────────
# Semicolon-delimited but not part of the have/want/admired pattern.
# e.g. AIFrustration, AIHuman, LearnCode, AIAgent_Uses
print("OTHER MULTI-SELECT examples:")
for c in other_multiselect_cols[:4]:
    print(f"\n--- {c} ---")
    df_bronze.select("ResponseId", c).filter(F.col(c).isNotNull()).show(3, truncate=80)

# Null Rates (Skip Logic)

Many survey columns are only filled when a prior answer qualifies the respondent (e.g., AI agent questions only if they use agents). High null rates are expected, not dirty data.

In [ ]:
total = df_bronze.count()

null_counts = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
])

null_row = null_counts.collect()[0]
null_rates = [(c, null_row[c], round(null_row[c] / total * 100, 1)) for c in df_bronze.columns]
null_rates.sort(key=lambda x: -x[2])

print(f"{'Column':<45} {'Nulls':>7} {'% Null':>7}")
print("-" * 62)
for col_name, n, pct in null_rates:
    if pct > 0:
        print(f"{col_name:<45} {n:>7,} {pct:>6.1f}%")

## Export scalar columns for collaborators

Run the **classification** cell above first so `scalar_cols` is defined. Writes one CSV under `exports/` at the project root (easy to share; ~49k rows fits in memory).

In [6]:
# Prerequisite: classification cell must have run (defines scalar_cols, df_bronze)
export_dir = Path.cwd().parents[1] / "exports"
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / "survey_2025_scalar_columns.csv"

df_scalar = df_bronze.select(*scalar_cols)
row_count = df_scalar.count()
print(f"Exporting {row_count:,} rows x {len(scalar_cols)} columns -> {export_path}")

df_scalar.toPandas().to_csv(export_path, index=False)
print("Done.")

Exporting 49,191 rows x 43 columns -> /Users/viktor/repos/stackoverflow-project/exports/survey_2025_scalar_columns.csv


Done.
